# dlcobra エラー分析 (Top-K Error Analysis)

モデルの予測エラーを分析する。
- 高確信度の誤り局面を抽出
- 将棋盤SVGで可視化
- スライス別（序盤/中盤/終盤）の精度確認

In [ ]:
import sys
import numpy as np
import torch
import cshogi
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

DEVICE = 'cpu'  # GPU学習中のためCPUを使用
MAX_SAMPLES = 200  # CPUなので少なめに
print(f'Device: {DEVICE}')

## 1. モデルのロード

In [ ]:
from dlshogi.experiments.exp026_droppath_bf16_fresh.model import PolicyValueNetwork

# exp026 checkpoint（現在学習中のrun: bz5nhn8j）
# 学習が進んだら新しいstepのものに変更
CKPT_PATH = '../dlshogi/wandb/wcsc36/bz5nhn8j/checkpoints/last.ckpt'

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
# torch.compile使用時はキーが '_orig_mod.' プレフィックスになる
state_dict = {k.replace('model._orig_mod.', '', 1): v 
              for k, v in ckpt['state_dict'].items() 
              if k.startswith('model._orig_mod.')}
if not state_dict:  # compile未使用の場合のフォールバック
    state_dict = {k.replace('model.', '', 1): v 
                  for k, v in ckpt['state_dict'].items() 
                  if k.startswith('model.')}

model = PolicyValueNetwork().to(DEVICE)
model.load_state_dict(state_dict)
model.eval()
print('Loaded:', CKPT_PATH)
print('step:', ckpt.get('global_step', 'unknown'))

## 2. SFENつきエラー収集

In [ ]:
from cshogi.dlshogi import make_input_features, make_move_label, FEATURES1_NUM, FEATURES2_NUM
import numpy as np

VAL_FILE = '/mnt/nvme1n1p2/data/shogi-ai-book/floodgate_test_2017-2018_r3500_eval5000.hcpe'
MAX_SAMPLES = 200  # CPUの場合は少なめに

hcpes = np.fromfile(VAL_FILE, dtype=cshogi.HuffmanCodedPosAndEval)
board = cshogi.Board()
sfen_errors = []
total = 0

for hcpe in hcpes[:MAX_SAMPLES]:
    board.set_hcp(hcpe['hcp'])
    true_move16 = hcpe['bestMove16']
    true_label = make_move_label(true_move16, board.turn)
    total += 1

    features1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
    features2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
    make_input_features(board, features1, features2)
    x1 = torch.tensor(features1).unsqueeze(0).to(DEVICE)
    x2 = torch.tensor(features2).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        policy, _ = model(x1, x2)
        prob = torch.softmax(policy, dim=1)
        pred_label = prob.argmax().item()
        confidence = prob.max().item()

    if pred_label != true_label:
        sfen_errors.append({
            'sfen': board.sfen(),
            'true_move16': true_move16,
            'pred_label': pred_label,
            'confidence': confidence,
            'move_number': board.move_number,
        })

sfen_errors.sort(key=lambda x: x['confidence'], reverse=True)
print(f'Accuracy: {1 - len(sfen_errors)/total:.4f} ({total - len(sfen_errors)}/{total})')
print(f'Errors: {len(sfen_errors)}')

## 3. 局面の可視化（高確信度の誤り）

In [ ]:
from IPython.display import display, SVG

def label_to_move_usi(board, label):
    """pred_labelから合法手のUSI文字列を逆引き"""
    for move in board.legal_moves:
        if make_move_label(cshogi.move16(move), board.turn) == label:
            return cshogi.move_to_usi(move)
    return f'label={label}(illegal)'

def show_error(err, idx):
    b = cshogi.Board(err['sfen'])
    pred_usi = label_to_move_usi(b, err['pred_label'])
    print(f"[{idx}] 手数:{err['move_number']}  確信度:{err['confidence']:.3f}")
    print(f"  正解手: {cshogi.move_to_usi(err['true_move16'])}  予測手: {pred_usi}")
    display(b.to_svg(lastmove=err['true_move16'], scale=1.05))

TOP_N = 10  # 表示件数
for i, err in enumerate(sfen_errors[:TOP_N]):
    show_error(err, i)

## 4. スライス分析（評価値）

In [ ]:
# スライス分析はサンプル数が多いほど正確。GPU空き時はMAX_SAMPLESを増やすこと（推奨: 2000+）
import matplotlib.pyplot as plt

# 手番側から見た評価値で6スライス
SLICES = ['Winning(>=1000)', 'Advantage(300-999)', 'Equal(0-299)',
          'Disadvantage(-1~-299)', 'Losing(-300~-999)', 'Lost(<=-1000)']

def eval_slice(eval_val, turn):
    v = int(eval_val) if turn == 0 else -int(eval_val)
    if v >= 1000:   return SLICES[0]
    elif v >= 300:  return SLICES[1]
    elif v >= 0:    return SLICES[2]
    elif v >= -299: return SLICES[3]
    elif v >= -999: return SLICES[4]
    else:           return SLICES[5]

slice_stats = {s: [0, 0] for s in SLICES}  # [errors, total]

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        s = eval_slice(hcpe['eval'], board.turn)
        slice_stats[s][1] += 1
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        if torch.softmax(policy, dim=1).argmax().item() != true_label:
            slice_stats[s][0] += 1

accs = [(t-e)/t if t>0 else 0 for e,t in slice_stats.values()]
colors = ['#1565C0','#42A5F5','#A5D6A7','#FFCC80','#EF9A9A','#B71C1C']

plt.figure(figsize=(10, 4))
bars = plt.bar(SLICES, accs, color=colors)
plt.ylim(0, 1)
plt.ylabel('Policy Accuracy')
plt.title('Policy Accuracy by Eval Slice (turn-relative)')
plt.xticks(rotation=15, ha='right')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

for s, (e, t) in slice_stats.items():
    if t > 0:
        print(f'{s:25s}: acc={(t-e)/t:.4f}  errors={e}/{t}')
    else:
        print(f'{s:25s}: no samples')

## 5. スライス分析（手番）

In [ ]:
turn_stats = {'Black(先手)': [0, 0], 'White(後手)': [0, 0]}

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        s = 'Black(先手)' if board.turn == 0 else 'White(後手)'
        turn_stats[s][1] += 1
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        if torch.softmax(policy, dim=1).argmax().item() != true_label:
            turn_stats[s][0] += 1

for s, (e, t) in turn_stats.items():
    print(f'{s}: acc={(t-e)/t:.4f}  n={t}  errors={e}')

## 6. スライス分析（持ち駒数）

In [ ]:
# 持ち駒合計数で局面の複雑さを分類
hand_stats = {'Few(0-4)': [0,0], 'Mid(5-9)': [0,0], 'Many(10+)': [0,0]}

def hand_slice(board):
    total = sum(board.pieces_in_hand[0]) + sum(board.pieces_in_hand[1])
    if total <= 4:  return 'Few(0-4)'
    elif total <= 9: return 'Mid(5-9)'
    else:           return 'Many(10+)'

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        s = hand_slice(board)
        hand_stats[s][1] += 1
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        if torch.softmax(policy, dim=1).argmax().item() != true_label:
            hand_stats[s][0] += 1

import matplotlib.pyplot as plt
slices = list(hand_stats.keys())
accs = [(t-e)/t if t>0 else 0 for e,t in hand_stats.values()]
plt.figure(figsize=(6, 4))
bars = plt.bar(slices, accs, color=['#66BB6A','#FFA726','#EF5350'])
plt.ylim(0, 1)
plt.ylabel('Policy Accuracy')
plt.title('Policy Accuracy by Hand Piece Count')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01, f'{acc:.3f}', ha='center')
plt.tight_layout()
plt.show()
for s, (e, t) in hand_stats.items():
    print(f'{s}: acc={(t-e)/t:.4f}  n={t}  errors={e}') if t>0 else print(f'{s}: no samples')

## 7. スライス分析（正解手の駒種）

In [ ]:
# 正解手を指した駒の種類別精度
PIECE_NAMES = {1:'Pawn(歩)', 2:'Lance(香)', 3:'Knight(桂)', 4:'Silver(銀)',
               5:'Bishop(角)', 6:'Rook(飛)', 7:'Gold(金)', 8:'King(玉)',
               9:'ProPawn(と)', 10:'ProLance(成香)', 11:'ProKnight(成桂)',
               12:'ProSilver(成銀)', 14:'ProBishop(馬)', 15:'ProRook(龍)'}

piece_stats = {v: [0, 0] for v in PIECE_NAMES.values()}
piece_stats['Drop(打)'] = [0, 0]

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        m16 = hcpe['bestMove16']
        if cshogi.move_is_drop(m16):
            s = 'Drop(打)'
        else:
            piece = board.piece(cshogi.move_from(m16))
            pt = piece % 16
            s = PIECE_NAMES.get(pt, f'Unknown({pt})')
        if s not in piece_stats:
            piece_stats[s] = [0, 0]
        piece_stats[s][1] += 1
        true_label = make_move_label(m16, board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        if torch.softmax(policy, dim=1).argmax().item() != true_label:
            piece_stats[s][0] += 1

# サンプルありのみ表示
valid = {s: v for s, v in piece_stats.items() if v[1] > 0}
valid = dict(sorted(valid.items(), key=lambda x: (x[1][1]-x[1][0])/x[1][1], reverse=True))

slices = list(valid.keys())
accs = [(t-e)/t for e,t in valid.values()]
counts = [t for e,t in valid.values()]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(slices, accs)
ax.set_ylim(0, 1)
ax.set_ylabel('Policy Accuracy')
ax.set_title('Policy Accuracy by Piece Type of Correct Move')
plt.xticks(rotation=30, ha='right')
for bar, acc, n in zip(bars, accs, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{acc:.2f}\n(n={n})', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. スライス分析（王手局面）

In [ ]:
check_stats = {'InCheck(王手あり)': [0, 0], 'NoCheck(王手なし)': [0, 0]}

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        s = 'InCheck(王手あり)' if board.is_check() else 'NoCheck(王手なし)'
        check_stats[s][1] += 1
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        if torch.softmax(policy, dim=1).argmax().item() != true_label:
            check_stats[s][0] += 1

for s, (e, t) in check_stats.items():
    print(f'{s}: acc={(t-e)/t:.4f}  n={t}  errors={e}') if t > 0 else None

## 9. Top-K Accuracy

In [ ]:
topk_correct = {1: 0, 3: 0, 5: 0}
total_topk = 0

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        topk5 = torch.topk(torch.softmax(policy, dim=1)[0], 5).indices.tolist()
        total_topk += 1
        for k in [1, 3, 5]:
            if true_label in topk5[:k]:
                topk_correct[k] += 1

import matplotlib.pyplot as plt
ks = [1, 3, 5]
accs = [topk_correct[k]/total_topk for k in ks]
plt.figure(figsize=(5, 4))
bars = plt.bar([f'Top-{k}' for k in ks], accs, color=['#EF5350','#FFA726','#66BB6A'])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Top-K Policy Accuracy')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{acc:.3f}', ha='center')
plt.tight_layout()
plt.show()
for k, acc in zip(ks, accs):
    print(f'Top-{k}: {acc:.4f} ({topk_correct[k]}/{total_topk})')

## 10. Confidence Calibration

In [ ]:
import matplotlib.pyplot as plt

N_BINS = 10
bins = np.linspace(0, 1, N_BINS+1)
bin_correct = np.zeros(N_BINS)
bin_total   = np.zeros(N_BINS)
bin_conf    = np.zeros(N_BINS)

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, _ = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        prob = torch.softmax(policy, dim=1)[0]
        conf = prob.max().item()
        pred = prob.argmax().item()
        b = min(int(conf * N_BINS), N_BINS-1)
        bin_total[b] += 1
        bin_conf[b]  += conf
        if pred == true_label:
            bin_correct[b] += 1

valid = bin_total > 0
avg_conf = np.where(valid, bin_conf / np.maximum(bin_total, 1), 0)
accuracy = np.where(valid, bin_correct / np.maximum(bin_total, 1), 0)
ece = np.sum(np.abs(accuracy - avg_conf) * bin_total / bin_total.sum())

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1],'--', color='gray', label='Perfect calibration')
ax.bar(avg_conf[valid], accuracy[valid], width=0.08, alpha=0.7, label='Model')
ax.set_xlabel('Confidence')
ax.set_ylabel('Accuracy')
ax.set_title(f'Confidence Calibration (ECE={ece:.4f})')
ax.legend()
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.show()

print(f'ECE (Expected Calibration Error): {ece:.4f}')
print(f'\n{"Bin":>12}  {"AvgConf":>8}  {"Accuracy":>8}  {"n":>5}  gap')
for i in range(N_BINS):
    if bin_total[i] > 0:
        gap = accuracy[i] - avg_conf[i]
        print(f'{bins[i]:.1f}-{bins[i+1]:.1f}  {avg_conf[i]:.4f}  {accuracy[i]:.4f}  {int(bin_total[i]):5d}  {gap:+.3f}')

## 11. Value Head Calibration（勝率予測精度）

In [ ]:
import matplotlib.pyplot as plt

pred_values = []
true_values = []

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        result = int(hcpe['gameResult'])
        turn = board.turn
        # 手番側から見た勝敗: 1=勝ち, 0=負け
        win = 1.0 if (result==1 and turn==0) or (result==2 and turn==1) else 0.0
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        _, value_logit = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        pred_values.append(torch.sigmoid(value_logit).item())
        true_values.append(win)

pred = np.array(pred_values)
true = np.array(true_values)
mse = np.mean((pred - true)**2)
print(f'MSE: {mse:.4f}  RMSE: {mse**0.5:.4f}')

# Calibration plot
N_BINS = 10
bins = np.linspace(0, 1, N_BINS+1)
avg_preds, actual_wins, ns = [], [], []
for i in range(N_BINS):
    mask = (pred >= bins[i]) & (pred < bins[i+1])
    n = mask.sum()
    if n > 0:
        avg_preds.append(pred[mask].mean())
        actual_wins.append(true[mask].mean())
        ns.append(n)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1],'--', color='gray', label='Perfect calibration')
ax.scatter(avg_preds, actual_wins, s=[n/3 for n in ns], alpha=0.8, label='Model (size=n)')
for x, y, n in zip(avg_preds, actual_wins, ns):
    ax.annotate(f'n={n}', (x, y), textcoords='offset points', xytext=(5,3), fontsize=7)
ax.set_xlabel('Predicted Win Rate')
ax.set_ylabel('Actual Win Rate')
ax.set_title(f'Value Head Calibration (RMSE={mse**0.5:.4f})')
ax.legend()
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.show()

print(f'\n{"Bin":>12}  {"AvgPred":>8}  {"ActualWin":>9}  {"n":>5}  gap')
for i in range(N_BINS):
    mask = (pred >= bins[i]) & (pred < bins[i+1])
    n = mask.sum()
    if n > 0:
        ap = pred[mask].mean()
        aw = true[mask].mean()
        print(f'{bins[i]:.1f}-{bins[i+1]:.1f}  {ap:.4f}  {aw:.4f}     {n:5d}  {aw-ap:+.3f}')

## 12. Value Head 勝率分布分析

In [ ]:
import matplotlib.pyplot as plt

pred_values, true_values = [], []
with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        result = int(hcpe['gameResult'])
        turn = board.turn
        win = 1.0 if (result==1 and turn==0) or (result==2 and turn==1) else 0.0
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        _, value_logit = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        pred_values.append(torch.sigmoid(value_logit).item())
        true_values.append(win)

pred = np.array(pred_values)
true = np.array(true_values)

# 予測勝率分布 vs 実際の勝敗分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左: 予測勝率のヒストグラム（勝ち/負けで色分け）
axes[0].hist(pred[true==1], bins=20, alpha=0.6, color='#1565C0', label='Actual Win')
axes[0].hist(pred[true==0], bins=20, alpha=0.6, color='#B71C1C', label='Actual Loss')
axes[0].axvline(0.5, color='gray', linestyle='--')
axes[0].set_xlabel('Predicted Win Rate')
axes[0].set_ylabel('Count')
axes[0].set_title('Predicted Win Rate Distribution')
axes[0].legend()

# 右: 予測勝率の全体分布
axes[1].hist(pred, bins=20, color='#42A5F5', alpha=0.8)
axes[1].axvline(pred.mean(), color='red', linestyle='--', label=f'mean={pred.mean():.3f}')
axes[1].axvline(0.5, color='gray', linestyle='--', label='0.5')
axes[1].set_xlabel('Predicted Win Rate')
axes[1].set_ylabel('Count')
axes[1].set_title('Overall Predicted Win Rate Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'予測勝率: mean={pred.mean():.4f}  std={pred.std():.4f}  median={np.median(pred):.4f}')
print(f'実際の勝率: {true.mean():.4f} (win={int(true.sum())}/{len(true)})')
print(f'予測0.4-0.6(中間帯): {((pred>=0.4)&(pred<=0.6)).mean()*100:.1f}%')
print(f'予測<0.1 or >0.9(確信帯): {((pred<0.1)|(pred>0.9)).mean()*100:.1f}%')

# 分離度: 勝ち局面と負け局面の予測値の差
print(f'\n勝ち局面の平均予測: {pred[true==1].mean():.4f}')
print(f'負け局面の平均予測: {pred[true==0].mean():.4f}')
print(f'分離度(差): {pred[true==1].mean() - pred[true==0].mean():.4f}')

## 13. Policy × Value 関係分析

In [ ]:
import matplotlib.pyplot as plt

policy_correct, policy_conf, value_pred, value_true = [], [], [], []

with torch.no_grad():
    for hcpe in hcpes[:MAX_SAMPLES]:
        board.set_hcp(hcpe['hcp'])
        true_label = make_move_label(hcpe['bestMove16'], board.turn)
        result = int(hcpe['gameResult']); turn = board.turn
        win = 1.0 if (result==1 and turn==0) or (result==2 and turn==1) else 0.0
        f1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
        f2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
        make_input_features(board, f1, f2)
        policy, value_logit = model(torch.tensor(f1).unsqueeze(0), torch.tensor(f2).unsqueeze(0))
        prob = torch.softmax(policy, dim=1)[0]
        policy_correct.append(int(prob.argmax().item() == true_label))
        policy_conf.append(prob.max().item())
        value_pred.append(torch.sigmoid(value_logit).item())
        value_true.append(win)

pc    = np.array(policy_correct)
pconf = np.array(policy_conf)
vp    = np.array(value_pred)
vt    = np.array(value_true)
vc    = ((vp > 0.5) == vt.astype(bool))
vconf = np.abs(vp - 0.5) * 2  # 0=不確か, 1=確信

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. クロス集計（パイチャート）
labels = ['P=OK V=OK', 'P=OK V=NG', 'P=NG V=OK', 'P=NG V=NG']
sizes  = [(pc==1)&(vc==1), (pc==1)&(vc==0), (pc==0)&(vc==1), (pc==0)&(vc==0)]
counts = [m.sum() for m in sizes]
colors = ['#1565C0','#42A5F5','#EF9A9A','#B71C1C']
axes[0].pie(counts, labels=[f'{l}\n{c}({c/len(pc)*100:.1f}%)' for l,c in zip(labels,counts)],
            colors=colors, startangle=90)
axes[0].set_title('Policy × Value Cross Table')

# 2. Policy確信度 vs Value確信度 散布図
axes[1].scatter(pconf[pc==1], vconf[pc==1], alpha=0.3, s=5, color='#1565C0', label='Policy OK')
axes[1].scatter(pconf[pc==0], vconf[pc==0], alpha=0.3, s=5, color='#B71C1C', label='Policy NG')
corr = np.corrcoef(pconf, vconf)[0,1]
axes[1].set_xlabel('Policy Confidence')
axes[1].set_ylabel('Value Confidence (|pred-0.5|×2)')
axes[1].set_title(f'Policy vs Value Confidence (r={corr:.3f})')
axes[1].legend(markerscale=3)

# 3. Policy正解/誤り別のValue予測分布
axes[2].hist(vp[pc==1], bins=20, alpha=0.6, color='#1565C0', label=f'Policy OK (n={pc.sum()})')
axes[2].hist(vp[pc==0], bins=20, alpha=0.6, color='#B71C1C', label=f'Policy NG (n={(pc==0).sum()})')
axes[2].axvline(0.5, color='gray', linestyle='--')
axes[2].set_xlabel('Predicted Win Rate')
axes[2].set_ylabel('Count')
axes[2].set_title('Value Prediction by Policy Result')
axes[2].legend()

plt.tight_layout()
plt.show()

print('=== Policy × Value クロス集計 ===')
for label, mask in zip(labels, sizes):
    print(f'{label}: {mask.sum():4d} ({mask.mean()*100:.1f}%)')

print(f'\n=== Policy確信度 vs Value確信度 ===')
print(f'相関係数: {corr:.4f}')
print(f'Policy高確信(>0.7)のValue確信度平均: {vconf[pconf>0.7].mean():.4f}')
print(f'Policy低確信(<0.3)のValue確信度平均: {vconf[pconf<0.3].mean():.4f}')

print(f'\n=== Policy誤り局面でのValue精度 ===')
print(f'Policy正解局面のValue正解率: {vc[pc==1].mean():.4f}  RMSE: {np.sqrt(np.mean((vp[pc==1]-vt[pc==1])**2)):.4f}')
print(f'Policy誤り局面のValue正解率: {vc[pc==0].mean():.4f}  RMSE: {np.sqrt(np.mean((vp[pc==0]-vt[pc==0])**2)):.4f}')